# wandb-config-into-args — ex2: three-tier precedence: defaults < wandb.config < cli_overrides

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `wandb-config-into-args`. Running the final beacon cell reports progress against the `Logging: wandb.config into args` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Logging: wandb.config into args` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`wandb-config-into-args`** (exercise 2). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "wandb-config-into-args"
DD_SUBTOPIC = "Logging: wandb.config into args"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## wandb.config + CLI overrides — three-tier precedence

Ex1 overwrote args from `wandb.config` unconditionally. Real ARENA sweep scripts have a THIRD tier: explicit CLI flags that should win over the sweep agent (debug-rerun a sweep config with a hand-tweaked lr).

Precedence (lowest → highest):
```
defaults  <  wandb.config (sweep)  <  cli_overrides (explicit)
```

**Same `setattr` + `hasattr` skeleton.** Apply each layer in order. The later layer wins on shared keys. Unknown keys (e.g. wandb metadata) still get silently skipped per ex1's contract.

**Why CLI beats sweep.** A sweep is automated exploration; a CLI override is a deliberate human intervention. Production sweep harnesses (Hydra, MMCV) all follow the same ordering.

### Exercise 2 — three-tier precedence: defaults < wandb.config < cli_overrides

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Bloom level: Apply
> LO: Apply chained `setattr(args, k, v) if hasattr(args, k)` in the order defaults → wandb.config → cli_overrides, so the highest-precedence layer's values are the ones that stick on the dataclass `args`.
> Keywords: wandb, sweep, cli, precedence
> ```

**KCs targeted:** `ordered-setattr-precedence`, `skip-unknown-keys-defensive`

Implement `ex2_apply_three_tier_config(args, sweep_cfg, cli_overrides)`. Ex1 collapsed sweep into args; ex2 adds a third tier on top.

Inputs:
- `args`: a dataclass instance with hparam fields.
- `sweep_cfg`: `dict` from `dict(wandb.config)` — may include wandb metadata keys (`'_wandb'`, `'_runtime'`) that aren't fields on args.
- `cli_overrides`: `dict` from argparse-style explicit user flags.

Algorithm:
1. For each `(k, v)` in `sweep_cfg.items()`: if `hasattr(args, k)`, `setattr(args, k, v)`. Else skip.
2. For each `(k, v)` in `cli_overrides.items()`: same — `setattr` only if `hasattr(args, k)`. Skip unknown.
3. Return the (mutated) `args`.

Precedence: cli_overrides applied LAST wins. The `defaults` tier is implicit in the dataclass field defaults — already on `args` at entry.

Return the SAME args instance (mutation contract from ex1).

In [ ]:
def ex2_apply_three_tier_config(args, sweep_cfg: dict, cli_overrides: dict):
    """Apply sweep then CLI on top of args; skip unknown keys; return args."""
    raise NotImplementedError()


def _test_ex2():
    from dataclasses import dataclass

    @dataclass
    class FakeArgs:
        lr: float = 1e-3
        batch_size: int = 64
        optimizer: str = 'adam'
        weight_decay: float = 0.0
        project: str = 'arena-default'

    # === Sweep alone overwrites defaults ===
    args = FakeArgs()
    out = ex2_apply_three_tier_config(args, {'lr': 5e-4, 'batch_size': 128}, {})
    assert out is args, 'must mutate-and-return the same instance'
    assert args.lr == 5e-4 and args.batch_size == 128
    assert args.optimizer == 'adam', 'untouched field stays default'

    # === CLI override BEATS sweep ===
    args = FakeArgs()
    ex2_apply_three_tier_config(
        args,
        {'lr': 5e-4, 'batch_size': 128},          # sweep
        {'lr': 1e-2},                              # CLI
    )
    assert args.lr == 1e-2, f'CLI lr (1e-2) must win over sweep lr (5e-4), got {args.lr}'
    assert args.batch_size == 128, 'batch_size from sweep (no CLI override) must persist'

    # === Sweep beats default; CLI beats both ===
    args = FakeArgs()
    ex2_apply_three_tier_config(
        args,
        {'weight_decay': 0.01},                   # sweep
        {'weight_decay': 0.0001},                  # CLI
    )
    assert args.weight_decay == 0.0001

    # === Unknown keys silently skipped in BOTH layers ===
    args = FakeArgs()
    ex2_apply_three_tier_config(
        args,
        {'lr': 5e-4, '_wandb': {'v': '0.17'}, 'mystery_a': 1},
        {'optimizer': 'sgd', '_runtime': 100, 'mystery_b': 2},
    )
    assert args.lr == 5e-4 and args.optimizer == 'sgd'
    assert not hasattr(args, '_wandb')
    assert not hasattr(args, 'mystery_a')
    assert not hasattr(args, '_runtime')
    assert not hasattr(args, 'mystery_b')

    # === Both dicts empty → args unchanged ===
    args = FakeArgs(lr=2e-3)
    ex2_apply_three_tier_config(args, {}, {})
    assert args.lr == 2e-3 and args.batch_size == 64

    # === Only CLI provided → CLI overrides defaults directly ===
    args = FakeArgs()
    ex2_apply_three_tier_config(args, {}, {'project': 'arena-cli'})
    assert args.project == 'arena-cli'

    # === Only sweep provided → sweep overrides defaults ===
    args = FakeArgs()
    ex2_apply_three_tier_config(args, {'project': 'arena-sweep'}, {})
    assert args.project == 'arena-sweep'

    # === CLI=None still overrides (explicit None choice) ===
    args = FakeArgs()
    ex2_apply_three_tier_config(args, {'lr': 5e-4}, {'lr': None})
    assert args.lr is None, f'CLI None must win, got {args.lr!r}'

    # === Inputs dicts are not mutated ===
    args = FakeArgs()
    sw = {'lr': 5e-4}
    cli = {'lr': 1e-2}
    ex2_apply_three_tier_config(args, sw, cli)
    assert sw == {'lr': 5e-4} and cli == {'lr': 1e-2}, 'input dicts must not be mutated'
    _dd_passed.add('ex2')
    print("ex2 ✓")

_test_ex2()

<details><summary>Solution</summary>

```python
def ex2_apply_three_tier_config(args, sweep_cfg, cli_overrides):
    for k, v in sweep_cfg.items():
        if hasattr(args, k):
            setattr(args, k, v)
    for k, v in cli_overrides.items():
        if hasattr(args, k):
            setattr(args, k, v)
    return args
```

**Order is the only thing that matters.** Both layers use the same `hasattr` + `setattr` skeleton; the precedence comes entirely from APPLICATION ORDER. Last write wins. Reversing the two `for` loops would make sweep beat CLI — the opposite of what production sweep harnesses expect.

**Why not merge dicts first, then setattr.** You COULD do `merged = {**sweep_cfg, **cli_overrides}; for k, v in merged...`. Same end state. The explicit two-pass form scales better when you add a fourth tier (env vars, file config) and is easier to step-debug.

**CLI=None as a legitimate override.** A user passing `--lr=None` explicitly is making a choice — the apply layer shouldn't filter it. If the downstream training code can't handle `lr=None`, that's a validation step at the dataclass `__post_init__`, not here.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex2'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex2',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()